# BIL545 · 4. Hafta · Colab defteri: veri ambarı ve veri küpleri

Veri Madenciliğinde İleri Konular · Dr. Selma Güzel · Üsküdar Üniversitesi

Bu defterde, sunumdaki sırayla: **yıldız şema** (olgu ve boyut tablolarını birleştirmek), **ölçü türleri** (dağıtılabilir, cebirsel, bütüncül), **kavram hiyerarşisi** ile **roll-up** ve **drill-down**, **slice, dice ve pivot**, **küboit kafesi** ve **Simpson paradoksu**. Sonda ders notundaki **zaman noktasına göre birleştirme** örneği.

Bu haftanın kodlarında en çok kullanılan araç `groupby`'dır. Her adımda **hangi sütunun, hangi gruplara göre, hangi işlemle** özetlendiğini ara adımlarla göstereceğiz.

**Bu defter nasıl kullanılır?**

* Hücreleri **yukarıdan aşağıya, sırayla** çalıştırın: hücreye tıklayıp **Shift + Enter**. Sonraki hücreler öncekilerde tanımlanan değişkenleri kullanır; ortadan başlarsanız `NameError` hatası alırsınız. Defteri baştan çalıştırmak için: Çalışma zamanı → Tümünü çalıştır.
* Sunumdaki her kod, burada **satır satır ya da birkaç satırlık parçalar hâlinde** verilmiştir. Her parçanın önünde iki açıklama vardır:
  * 🐍 **Python:** satırdaki Python kavramı (Python bilmeniz gerekmez).
  * 📊 **Konu:** satırın veri madenciliği açısından ne yaptığı ve neden yapıldığı.
* 🔍 **Ara adım** hücreleri sunumda yoktur: bir hesabın neye uygulandığını görebilmeniz için ara sonucu gösterir (ör. ortalamanın hangi sütunun hangi değerlerine uygulandığı).
* Her bölümün sonunda **sunumdaki hâli** tek hücre olarak tekrar verilir; çıktısı sunumdakiyle aynı olmalıdır. ✅ **Sonucu okuyalım** kutusu sonucu yorumlar.
* Kodda `#` işaretinden sonraki her şey **yorumdur**: Python onu çalıştırmaz, insanlar için yazılmış nottur.

### `groupby` nasıl okunur?

Bu haftanın hemen her satırı şu kalıptadır:

```python
tablo.groupby("GRUP_SÜTUNU")["ÖLÇÜ_SÜTUNU"].İŞLEM()
```

Soldan sağa okuyun:
1. `tablo.groupby("GRUP_SÜTUNU")`: satırları, grup sütununun değerlerine göre **gruplara ayır** (ör. her ilçe bir grup).
2. `["ÖLÇÜ_SÜTUNU"]`: her grupta **hangi sütunun** değerleriyle ilgilendiğimizi seç (ör. tutar).
3. `.İŞLEM()`: o değerlere her grupta **ayrı ayrı** uygulanacak özet: `sum()` toplam, `mean()` ortalama, `count()` sayım, `median()` medyan.

Sonuç: her grup için **tek bir sayı**. Yani `mean` tablonun tamamına değil, **her grubun seçilen sütunundaki değerlere** uygulanır. Bu üç adım, veri küpünün tanımıdır: **boyut** (grup sütunu), **ölçü** (seçilen sütun) ve **özet işlemi**.

## 1. Yıldız şema: olgu ve boyut tablolarını birleştirmek

**Konu.** Ambarda veri **yıldız şema** ile saklanır: ortada bir **olgu tablosu** (her satır bir olay, ör. bir satış; içinde **ölçüler** ve boyutlara bağlanan **anahtarlar**), çevresinde **boyut tabloları** (zaman, ürün, şube; her biri o açının ayrıntılarını tutar). Analiz için olgu tablosu boyut tablolarıyla **birleştirilir**; sonra boyutlara göre özetlenir.

Burada küçük, sentetik bir eczane örneği kuruyoruz: 6 satış, 3 boyut (zaman, ürün, şube), 2 ölçü (adet, tutar).

🐍 **Python:** pandas modülünü `pd` takma adıyla açar.

In [ ]:
import pandas as pd

🐍 **Python:** **Sözlükten tablo:** süslü parantez bir **sözlüktür**; her **anahtar** (`"tarih_id"`, …) bir sütun adı, her değer o sütunun değerlerini tutan bir **liste**dir. Listeler aynı uzunlukta olmalı (6 satış). `pd.DataFrame(...)` bunu tabloya çevirir. Komut, parantezler kapanana kadar birden çok satıra yayılabilir.

📊 **Konu:** Olgu tablosu. İlk üç sütun **anahtardır** (yabancı anahtar): hangi tarihte, hangi ürün, hangi şube. Son iki sütun **ölçüdür**: özetlenecek sayılar. Olgu tablosunda ürün adı ya da ilçe adı yoktur; yalnız numaraları vardır.

In [ ]:
# Olgu tablosu: her satır bir satış (ölçüler: adet, tutar)
olgu = pd.DataFrame({
    "tarih_id": [1, 1, 2, 2, 3, 3],
    "urun_id":  [10, 20, 10, 30, 20, 30],
    "sube_id":  [100, 100, 200, 200, 100, 200],
    "adet":     [3, 1, 2, 5, 2, 1],
    "tutar":    [450, 120, 300, 250, 240, 50],
})

🔍 **Ara adım:** Olgu tablosunu görelim. Hücrenin son satırında yalnız bir tablo adı varsa Colab onu tablo olarak gösterir.

In [ ]:
olgu

🐍 **Python:** Aynı kalıpla üç küçük tablo. `urun` tablosunun tanımı iki satıra yayılmış.

📊 **Konu:** Üç **boyut tablosu**. Her birinin ilk sütunu **anahtardır** (olgu tablosundaki numarayla eşleşir); diğer sütunlar o boyutun ayrıntıları ve **kavram hiyerarşisinin** düzeyleridir: zaman için tarih → ay → çeyrek, ürün için ürün → kategori, şube için ilçe → il.

In [ ]:
# Boyut tabloları
zaman = pd.DataFrame({"tarih_id": [1, 2, 3], "ay": ["Ocak", "Ocak", "Şubat"], "ceyrek": ["Ç1", "Ç1", "Ç1"]})
urun  = pd.DataFrame({"urun_id": [10, 20, 30], "urun": ["Tansiyon aleti", "Vitamin", "Maske"],
                      "kategori": ["Cihaz", "Takviye", "Sarf"]})
sube  = pd.DataFrame({"sube_id": [100, 200], "ilce": ["Üsküdar", "Kadıköy"], "il": ["İstanbul", "İstanbul"]})

🔍 **Ara adım:** **Birleştirmeyi adım adım görelim.** `merge(..., on="tarih_id")` iki tabloyu ortak sütundaki **eşleşen değerlere** göre yan yana koyar (SQL'deki JOIN). Olgu tablosunun her satırına, `tarih_id`'si aynı olan zaman satırının `ay` ve `ceyrek` bilgisi eklenir. 1 numaralı tarihli iki satış "Ocak" alır.

In [ ]:
olgu.merge(zaman, on="tarih_id")

🔍 **Ara adım:** İkinci birleştirme: şimdi her satıra ürün adı ve kategorisi eklendi. **Zincirleme:** ilk `merge`'ün sonucu bir tablodur; ikinci `merge` onun üzerinde çalışır.

In [ ]:
olgu.merge(zaman, on="tarih_id").merge(urun, on="urun_id")

🐍 **Python:** Arka arkaya üç `merge`: olgu tablosu üç boyut tablosuyla birleştirilir; sonuç `tablo` adına atanır.

📊 **Konu:** Artık her satış, bütün boyut bilgileriyle tek satırda. Analiz bu birleşik tablo üzerinde yapılır.

In [ ]:
tablo = olgu.merge(zaman, on="tarih_id").merge(urun, on="urun_id").merge(sube, on="sube_id")

🐍 **Python:** `tablo[[...]]` (çift köşeli parantez): yalnız listedeki sütunları seçer. İçteki köşeli parantez bir **liste**, dıştaki seçim işaretidir.

📊 **Konu:** Okunur bir görünüm: hangi ayda, hangi ürün, hangi ilçede, kaç adet, kaç TL.

In [ ]:
print(tablo[["ay", "urun", "ilce", "adet", "tutar"]])

🔍 **Ara adım:** **groupby'ı adım adım görelim.** Tek başına `groupby` bir sonuç göstermez; yalnız "gruplanmış" bir nesne verir. Henüz hiçbir şey hesaplanmadı.

In [ ]:
tablo.groupby("ilce")

🔍 **Ara adım:** Grupların içini görelim: döngü her grup için grubun adını (`ilce`) ve o gruba düşen satırları (`grup`) verir. Kadıköy'e 3 satış, Üsküdar'a 3 satış düştü.

In [ ]:
for ilce, grup in tablo.groupby("ilce"):
    print(ilce)
    print(grup[["urun", "tutar"]])
    print()

🐍 **Python:** Üç adım: `groupby("ilce")` satırları ilçeye göre gruplar → `["tutar"]` her grupta tutar sütununu seçer → `.sum()` her grubun tutarlarını **toplar**.

📊 **Konu:** **Boyut:** ilçe. **Ölçü:** tutar. **Özet:** toplam. Bu, şube boyutunda ilçe düzeyinde bir özettir.

In [ ]:
print(tablo.groupby("ilce")["tutar"].sum())

✅ **Sonucu okuyalım:** Kadıköy: 300 + 250 + 50 = **600** TL; Üsküdar: 450 + 120 + 240 = **810** TL. Ambarlarda olgu tablosu milyonlarca satırdır, boyut tabloları ise küçük kalır; ama işlem aynıdır: **birleştir, grupla, özetle**.

🔍 **Ara adım:** Deneyin: aynı özet **il** düzeyinde. İki ilçe de İstanbul'da olduğu için tek bir satır (1410) çıkar. Bu, şube hiyerarşisinde bir düzey yukarı çıkmaktır: **roll-up**.

In [ ]:
tablo.groupby("il")["tutar"].sum()

## 2. Ölçü türleri: parçalardan bütün hesaplanabilir mi?

**Konu.** Bir küp parça parça hesaplanır: önce şube şube, sonra il il. Bir ölçünün türü, **parçaların özetlerinden bütünün özetinin çıkıp çıkamayacağını** söyler:

| Tür | Tanım | Örnek |
|---|---|---|
| **Dağıtılabilir** | Parçaların sonuçları aynı işlemle birleştirilince bütünün sonucu çıkar. | toplam, sayım, en küçük, en büyük |
| **Cebirsel** | Parçalardan birkaç dağıtılabilir değer saklanırsa hesaplanır. | ortalama = toplam / sayım |
| **Bütüncül** | Parçaların sabit sayıda özeti yetmez; bütün veri gerekir. | medyan, mod, farklı değer sayısı |

**Veri: credit-g.** Bir Alman bankasına yapılmış 1000 kredi başvurusu (Weka'daki `credit-g.arff`'in aynısı). Her başvuru iyi (`good`) ya da kötü (`bad`) etiketli. Burada **ölçü**: kredi tutarı (`credit_amount`). **Parçalar**: konut durumu (`housing`): kirada (`rent`), ev sahibi (`own`), ücretsiz oturan (`for free`).

🐍 **Python:** `fetch_openml`: OpenML'deki bir veri kümesini adıyla indiren işlev.

In [ ]:
from sklearn.datasets import fetch_openml

🐍 **Python:** İşlev bir sonuç nesnesi döndürür; `.frame` verinin tablo hâlidir; `df` adına atanır.

📊 **Konu:** 1000 satır, 21 sütun (20 öznitelik ve hedef `class`).

In [ ]:
df = fetch_openml("credit-g", version=1, as_frame=True).frame   # Weka'daki aynı veri

🔍 **Ara adım:** Bu defterde kullanacağımız sütunlara bakalım: konut durumu, kredi tutarı, kredi amacı, yaş, sınıf.

In [ ]:
print(df.shape)
df[["housing", "credit_amount", "purpose", "age", "class"]].head()

🐍 **Python:** Üç adımlı kalıbın ilk iki adımı: satırları `housing`'e göre grupla, her grupta `credit_amount` sütununu seç. **Henüz bir işlem uygulanmadı**; sonuç gruplanmış bir sütundur ve `parca` adına atanır. Böylece aynı gruplamaya birazdan farklı işlemler (`sum`, `count`, `mean`, `median`) uygulayabileceğiz.

`observed=True`: kategorik sütunlarda yalnız veride **gerçekten geçen** kategorilere göre gruplar. Yazılmazsa Colab bir uyarı verir.

📊 **Konu:** Üç **parça**: verinin konut durumuna göre üç alt kümesi.

In [ ]:
parca = df.groupby("housing", observed=True)["credit_amount"]   # 3 parça: rent, own, for free

🐍 **Python:** **Çoklu atama:** sağdaki iki sonuç sırayla soldaki iki ada atanır. `parca.sum()` her parçanın kredi tutarlarının **toplamı**, `parca.count()` her parçadaki **kayıt sayısı**.

📊 **Konu:** Her parçadan iki **dağıtılabilir** değer saklıyoruz: toplam ve sayım.

In [ ]:
toplamlar, sayilar = parca.sum(), parca.count()

🔍 **Ara adım:** Her parça için toplam tutar ve kayıt sayısı. Sayılar: rent 179, own 713, for free 108 (toplam 1000).

In [ ]:
print(toplamlar)
print(sayilar)

### a) Toplam: dağıtılabilir

🐍 **Python:** Soldaki: üç parça toplamının toplamı (`toplamlar` üç sayılık bir sütun; `.sum()` onları toplar). Sağdaki: bütün verinin kredi tutarı toplamı (gruplama yok).

📊 **Konu:** İki sayı **aynı** çıkmalı: toplam, parçalardan doğru hesaplanır.

In [ ]:
print(toplamlar.sum(), df["credit_amount"].sum())               # dağıtılabilir

### b) Ortalama: cebirsel

🔍 **Ara adım:** Önce her parçanın ortalamasına bakalım: rent 3122,6; own 3060,9; for free 4906,2. `for free` grubunun ortalaması çok yüksek ama yalnız 108 kayıt var.

In [ ]:
print(parca.mean().round(1))

🐍 **Python:** Tek `print`, üç sayı; parantez açık olduğu için komut üç satıra yayılabilir. `round(x, 1)` bir ondalık basamağa yuvarlar.
* `parca.mean().mean()`: **iki ayrı** `mean`. İlki her parçanın ortalamasını alır (üç sayı), ikincisi bu **üç sayının** ortalamasını alır.
* `toplamlar.sum() / sayilar.sum()`: bütün toplam ÷ bütün sayım.
* `df["credit_amount"].mean()`: bütün verinin ortalaması (gruplama yok).

📊 **Konu:** Ortalama cebirsel bir ölçüdür: doğru birleştirmek için her parçanın **toplamını ve sayımını** saklamak gerekir; ortalamaları saklamak yetmez.

In [ ]:
print(round(parca.mean().mean(), 1),                            # ortalamaların ortalaması
      round(toplamlar.sum() / sayilar.sum(), 1),                # cebirsel: toplam / sayım
      round(df["credit_amount"].mean(), 1))                     # gerçek ortalama

### c) Medyan: bütüncül

🔍 **Ara adım:** Her parçanın medyanı: rent 2301; own 2238; for free 3800,5.

In [ ]:
print(parca.median())

🐍 **Python:** Yine iki ayrı işlem: `parca.median()` üç parçanın medyanı (üç sayı), ikinci `.median()` bu **üç sayının** medyanı. Sağdaki bütün verinin medyanı.

📊 **Konu:** Medyanların medyanı genel medyanı vermez.

In [ ]:
print(parca.median().median(), df["credit_amount"].median())    # bütüncül

**Sunumdaki hâli** (tek hücre):

In [ ]:
from sklearn.datasets import fetch_openml
df = fetch_openml("credit-g", version=1, as_frame=True).frame   # Weka'daki aynı veri

parca = df.groupby("housing", observed=True)["credit_amount"]   # 3 parça: rent, own, for free
toplamlar, sayilar = parca.sum(), parca.count()

print(toplamlar.sum(), df["credit_amount"].sum())               # dağıtılabilir
print(round(parca.mean().mean(), 1),                            # ortalamaların ortalaması
      round(toplamlar.sum() / sayilar.sum(), 1),                # cebirsel: toplam / sayım
      round(df["credit_amount"].mean(), 1))                     # gerçek ortalama
print(parca.median().median(), df["credit_amount"].median())    # bütüncül

✅ **Sonucu okuyalım:** * **Toplam:** iki sayı aynı (3 271 258): **dağıtılabilir**.
* **Ortalama:** ortalamaların ortalaması **3696,6** (yanlış); toplam / sayım **3271,3** (doğru, gerçek ortalamayla aynı). Ortalamaların ortalaması üç parçaya **eşit ağırlık** verir; oysa `for free` yalnız 108, `own` 713 kayıt. Küçük ve pahalı `for free` grubu sonucu yukarı çekiyor. **Cebirsel.**
* **Medyan:** medyanların medyanı **2301**, gerçek medyan **2319,5**. Parçalardan kurtarılamaz: **bütüncül**. Bu yüzden medyan küpleri pahalıdır.

(Colab'daki sürüme göre toplam `3271258` ya da `3271258.0` biçiminde yazılabilir; değer aynıdır.)

## 3. Kavram hiyerarşisi, roll-up ve drill-down

**Konu.** **Kavram hiyerarşisi**, bir boyutun ayrıntıdan genele düzeyleridir. `purpose` (kredi amacı) 10 değer alır; bunları üç gruba toplayan bir hiyerarşi kuruyoruz: yeni araç ve ikinci el araç → **araç**; mobilya, radyo/TV ve ev aleti → **ev eşyası**; geri kalanlar → **diğer**. Yaş için de sayısal bir hiyerarşi: yaş → yaş grubu.

* **Roll-up** (yukarı özetleme): hiyerarşide yukarı çıkmak (amaç → amaç grubu).
* **Drill-down** (ayrıntıya inme): aşağı inmek (amaç grubu → amaç).

**Ölçü:** kötü kredi oranı.

🐍 **Python:** Bir **sözlük**: her anahtar bir alt düzey değeri (amaç), her değer onun üst düzeyi (amaç grubu). Sözlük burada bir **eşleme tablosu** gibi çalışır.

📊 **Konu:** Kavram hiyerarşisini kodda böyle yazarız. "Diğer" grubundakileri (repairs, education, …) tek tek yazmıyoruz; bir sonraki satırda toplu olarak "diğer" yapılacak. Hiyerarşi bir **karardır**: "radyo/TV ev eşyası mıdır?" sorusunun cevabını alan bilgisi verir.

In [ ]:
grup = {"new car": "araç", "used car": "araç",
        "furniture/equipment": "ev eşyası", "radio/tv": "ev eşyası", "domestic appliance": "ev eşyası"}

🔍 **Ara adım:** **`map`'i adım adım görelim.** İlk satır ilk 8 başvurunun amacı. İkinci satır `map(grup)` sonrası: her değer sözlükteki karşılığıyla değiştirildi. Sözlükte **olmayan** değerler (ör. `education`, `business`) **NaN** (eksik) oldu.

In [ ]:
print(df["purpose"].astype(str).head(8).tolist())
print(df["purpose"].astype(str).map(grup).head(8).tolist())

🐍 **Python:** Zincirleme, soldan sağa: `astype(str)` kategoriyi düz metne çevirir → `map(grup)` her değeri sözlükteki karşılığıyla değiştirir, sözlükte olmayanlar NaN olur → `fillna("diğer")` NaN'ları "diğer" ile doldurur. Sonuç `df`'e **yeni bir sütun** (`amac_grubu`) olarak eklenir.

📊 **Konu:** Yeni boyut düzeyi: amaç grubu.

In [ ]:
df["amac_grubu"] = df["purpose"].astype(str).map(grup).fillna("diğer")          # amaç → amaç grubu

🔍 **Ara adım:** `value_counts()` her değerden kaç tane olduğunu sayar: ev eşyası 473, araç 337, diğer 190.

In [ ]:
print(df["amac_grubu"].value_counts())

🐍 **Python:** `pd.cut` sayıyı **aralıklara** böler. `bins=[18, 30, 45, 75]` sınırlardır: (18, 30], (30, 45], (45, 75]. Yuvarlak parantez "dâhil değil", köşeli "dâhil" demektir; 30 yaş ilk gruba girer. `labels` her aralığın adı.

📊 **Konu:** Sayısal bir kavram hiyerarşisi: yaş → yaş grubu (3. haftadaki ayrıklaştırma). Verideki yaşlar 19–75 arasında.

In [ ]:
df["yas_grubu"] = pd.cut(df["age"], bins=[18, 30, 45, 75], labels=["19–30", "31–45", "46–75"])

🔍 **Ara adım:** Her yaşın hangi gruba düştüğünü görelim.

In [ ]:
print(df[["age", "yas_grubu"]].head())

🐍 **Python:** `df["class"] == "bad"` her satır için True/False üretir; `astype(int)` bunları 1/0 yapar.

📊 **Konu:** Ölçü sütunu: kötü kredi 1, iyi kredi 0. **Önemli:** 0/1 bir sütunun **ortalaması bir orandır**: 1'lerin sayısı ÷ toplam. Bu yüzden aşağıda `mean()` gördüğünüz her yerde sonuç "o gruptaki kötü kredi oranı"dır.

In [ ]:
df["kotu"] = (df["class"] == "bad").astype(int)                                 # ölçü: kötü = 1

🔍 **Ara adım:** Kontrol: 300 kötü kredi, 1000 başvuru; ortalama = 300 / 1000 = 0,3. Yani `mean`, kötü kredi **oranını** verdi.

In [ ]:
print(df["kotu"].sum(), len(df), df["kotu"].mean())

🔍 **Ara adım:** Roll-up'tan önce bir grubu elle hesaplayalım: araç kredileri 337; bunların 106'sı kötü; 106 / 337 = **0,315**. Bir sonraki satırda `groupby` bunu üç grup için birden yapacak.

In [ ]:
ornek = df[df["amac_grubu"] == "araç"]
print(ornek["kotu"].sum(), len(ornek), ornek["kotu"].mean().round(3))

🐍 **Python:** Kalıp: `amac_grubu`'na göre grupla → `kotu` sütununu seç → her grubun **ortalaması** (= kötü oranı) → üç basamağa yuvarla.

📊 **Konu:** **Roll-up:** amaç grubu düzeyinde kötü kredi oranı.

In [ ]:
print(df.groupby("amac_grubu")["kotu"].mean().round(3))                         # roll-up

🐍 **Python:** `groupby`'a bir **liste** verilince iki düzeyli gruplar oluşur: önce amaç grubuna, onun içinde amaca göre. Çıktıda iki düzeyli satır etiketi görünür. `observed=True`, veride hiç geçmeyen `vacation` amacını gruplardan çıkarır.

📊 **Konu:** **Drill-down:** aynı özet bir düzey aşağıda.

In [ ]:
print(df.groupby(["amac_grubu", "purpose"], observed=True)["kotu"].mean().round(3))   # drill-down

✅ **Sonucu okuyalım:** Roll-up: araç **0,315**, diğer **0,368**, ev eşyası **0,262**. Drill-down: araç grubunun 0,315'i, birbirinden çok farklı iki oranın karışımı: yeni araç **0,380**, ikinci el araç **0,165**. Yukarı çıkmak her zaman **ayrıntı kaybıdır**; kaybın önemli olup olmadığı kararın sorusuna bağlıdır. Weka'da MergeManyValues ile yaptığımız birleştirmenin kesin sayıları bunlardır.

## 4. Slice, dice ve pivot

**Konu.**
* **İki boyutlu görünüm** (bir küboit): satırlarda bir boyut, sütunlarda başka bir boyut, hücrelerde ölçü.
* **Slice** (dilim): bir boyutta **tek bir değer** seçmek (ör. yalnız araç kredileri).
* **Dice** (alt küp): **birden çok boyutta** koşul koymak (ör. araç **ve** 19–30 yaş).
* **Pivot** (döndürme): satır ve sütun eksenlerinin yerini değiştirmek; sayılar aynı kalır.

🐍 **Python:** `pivot_table` iki boyutlu bir özet tablo kurar:
* `index="amac_grubu"`: **satırlara** amaç grubu,
* `columns="yas_grubu"`: **sütunlara** yaş grubu,
* `values="kotu"`: hücrelerde özetlenecek **ölçü sütunu**,
* `aggfunc="mean"`: özet işlemi: ortalama.

Yani her hücre: "bu amaç grubunda **ve** bu yaş grubundaki başvuruların `kotu` değerlerinin ortalaması" = o hücredeki kötü kredi oranı.

📊 **Konu:** İki boyutlu bir **küboit**: amaç grubu × yaş grubu.

In [ ]:
kup = df.pivot_table(index="amac_grubu", columns="yas_grubu", values="kotu",
                     aggfunc="mean", observed=True).round(3)

🔍 **Ara adım:** Aynı tablo, ama `aggfunc="count"`: her hücrede **kaç başvuru** olduğu. Bir oranı yorumlarken arkasındaki kayıt sayısına bakın: küçük hücrelerdeki oranlar oynaktır.

In [ ]:
df.pivot_table(index="amac_grubu", columns="yas_grubu", values="kotu", aggfunc="count", observed=True)

🐍 **Python:** Tabloyu yazdırır.

In [ ]:
print(kup)                                                            # 2 boyutlu görünüm

🐍 **Python:** **Mantıksal filtreleme**, içten dışa: `df["amac_grubu"] == "araç"` her satır için True/False üretir; bu True/False dizisi `df[...]` içine konunca yalnız **True** olan satırlar kalır. Sonuç `dilim` adına atanır; `df` değişmez.

📊 **Konu:** **Slice:** amaç grubu boyutunda tek bir değer (araç). 337 başvuru kalır.

In [ ]:
dilim = df[df["amac_grubu"] == "araç"]                                # slice

🔍 **Ara adım:** Dilimde kaç kayıt var ve medeni durum dağılımı nasıl? Weka'da RemoveWithValues sonrası `personal_status` histogramında gördüğümüz sayılar.

In [ ]:
print(len(dilim))
print(dilim["personal_status"].value_counts())

🐍 **Python:** Kalıp, bu kez **dilim** üzerinde: medeni duruma göre grupla → `kotu` → ortalama.

📊 **Konu:** Araç dilimi içinde medeni duruma göre kötü kredi oranı.

In [ ]:
print(dilim.groupby("personal_status", observed=True)["kotu"].mean().round(3))

🐍 **Python:** İki koşul `&` ("ve") ile birleştirilir: iki koşulu da sağlayan satırlar kalır. Her koşul **parantez içinde** yazılmalıdır. `and` sözcüğü burada çalışmaz; tablolarda `&` kullanılır.

📊 **Konu:** **Dice:** iki boyutta koşul: amaç grubu = araç **ve** yaş grubu = 19–30.

In [ ]:
zar = df[(df["amac_grubu"] == "araç") & (df["yas_grubu"] == "19–30")]   # dice

🐍 **Python:** `len(zar)` alt küpteki kayıt sayısı; `zar["kotu"].mean()` bu kayıtların kötü oranı (burada gruplama yok, tek bir sayı).

📊 **Konu:** Bu, `kup` tablosunun sol üst hücresinin (araç, 19–30) arkasındaki kayıtlardır; oran aynı olmalı.

In [ ]:
print(len(zar), zar["kotu"].mean().round(3))

🐍 **Python:** `.T` (*transpose*) tabloyu çevirir: satırlar sütun, sütunlar satır olur.

📊 **Konu:** **Pivot:** satırlarda yaş grubu, sütunlarda amaç grubu. Sayılar değişmez, okuma yönü değişir.

In [ ]:
print(kup.T)                                                          # pivot

**Sunumdaki hâli** (tek hücre):

In [ ]:
kup = df.pivot_table(index="amac_grubu", columns="yas_grubu", values="kotu",
                     aggfunc="mean", observed=True).round(3)
print(kup)                                                            # 2 boyutlu görünüm

dilim = df[df["amac_grubu"] == "araç"]                                # slice
print(dilim.groupby("personal_status", observed=True)["kotu"].mean().round(3))

zar = df[(df["amac_grubu"] == "araç") & (df["yas_grubu"] == "19–30")]   # dice
print(len(zar), zar["kotu"].mean().round(3))

print(kup.T)                                                          # pivot

✅ **Sonucu okuyalım:** * **İki boyutlu görünüm:** genç başvuranlarda (19–30) kötü oranı her amaç grubunda daha yüksek; en büyük fark araç kredilerinde (0,417 ile 0,248).
* **Slice:** araç diliminde male single **0,265** (204 kayıt, 54 kötü), female div/dep/mar **0,426**. Weka'daki araç dilimiyle aynı sayılar: aynı veri, aynı dilim, iki araç.
* **Dice:** araç ve 19–30: **120** kayıt, kötü oranı **0,417**: tablonun sol üst hücresi.
* **Pivot:** aynı sayılar, eksenler yer değiştirmiş.

## 5. Küboit kafesinin bütün katları

**Konu.** **Küboit**, küpün boyutların bir alt kümesine göre özetlenmiş hâlidir. Üç boyut (amaç grubu, yaş grubu, konut durumu) için:
* 0 boyutlu: **tepe küboit**, tek hücre: genel kötü oranı,
* 1 boyutlu: 3 küboit,
* 2 boyutlu: 3 küboit,
* 3 boyutlu: **temel küboit**, en ayrıntılı.

Toplam 2³ = 8 küboit. Kod bunların hepsini sırayla hesaplıyor ve her birinde kaç hücre olduğunu yazıyor.

🐍 **Python:** `itertools` Python'un hazır modülüdür; `combinations(liste, k)` listeden **k elemanlı bütün alt kümeleri** sırayla verir.

📊 **Konu:** Küboitler, boyut kümesinin alt kümeleridir; bu işlev onları üretiyor.

In [ ]:
from itertools import combinations

🐍 **Python:** Üç sütun adından oluşan bir liste.

📊 **Konu:** Küpün üç boyutu.

In [ ]:
boyutlar = ["amac_grubu", "yas_grubu", "housing"]

🔍 **Ara adım:** `combinations(boyutlar, 2)`: iki elemanlı bütün alt kümeler. Sonuçları görmek için `list(...)` ile listeye çeviriyoruz. Her alt küme bir **demettir** (tuple): parantezli, değiştirilemeyen bir liste. İkinci satır: `len(boyutlar)` 3; `range(4)` 0, 1, 2, 3 sayılarını verir (Python saymaya 0'dan başlar, son sayıyı dâhil etmez).

In [ ]:
print(list(combinations(boyutlar, 2)))
print(list(range(len(boyutlar) + 1)))

🔍 **Ara adım:** Döngünün içindeki hesabın **bir örneği**: iki boyutlu (amaç grubu, yaş grubu) küboit. Her hücre bir (amaç grubu, yaş grubu) çifti ve oradaki kötü oranı; 3 × 3 = 9 hücre. Döngü bunu 8 küboitin her biri için yapacak ve yalnız hücre sayısını yazacak.

In [ ]:
kuboit = df.groupby(["amac_grubu", "yas_grubu"], observed=True)["kotu"].mean()
print(kuboit.round(3))
print(len(kuboit), "hücre")

🐍 **Python:** **İç içe döngü:** dıştaki döngü k = 0, 1, 2, 3 için çalışır; her k için içteki döngü k elemanlı bütün alt kümeleri (`grup_by`) dolaşır.

**`if … else`:** koşul doğruysa (`k == 0`, yani hiç boyut yok) ilk blok, değilse ikinci blok çalışır. `==` karşılaştırmadır (eşit mi?), `=` atamadır. Hangi satırın hangi bloğa ait olduğunu **girinti** belirler.

`list(grup_by)`: `combinations` bir demet verir; `groupby` bir liste beklediği için listeye çeviriyoruz.

📊 **Konu:** `k == 0` iken gruplama yoktur: bütün verinin kötü oranı (**tepe küboit**). Diğer durumlarda o boyutlara göre gruplanmış kötü oranı hesaplanır ve hücre sayısı yazılır.

In [ ]:
for k in range(len(boyutlar) + 1):                    # 0, 1, 2, 3 boyutlu küboitler
    for grup_by in combinations(boyutlar, k):
        if k == 0:
            print("()", "→ 1 hücre:", round(df["kotu"].mean(), 3))      # tepe küboit
        else:
            kuboit = df.groupby(list(grup_by), observed=True)["kotu"].mean()
            print(grup_by, "→", len(kuboit), "hücre")

✅ **Sonucu okuyalım:** Tepe küboit: **0,3** (1 hücre). Tek boyutlu küboitler 3'er hücre, iki boyutlular 9'ar, temel küboit **27** hücre (3 × 3 × 3; bu veride hepsi dolu). Toplam 8 = 2³ küboit.

Hiyerarşiler de hesaba katılırsa küboit sayısı (L₁ + 1) × (L₂ + 1) × … × (Lₙ + 1) olur. Boyut sayısı arttıkça hücrelerin çoğu boş kalır: küpler **seyrektir**. Bu yüzden uygulamada yalnız bazı küboitler önceden hesaplanır (kısmi materyalleştirme) ya da yalnız yeterince kayıt içeren hücreler saklanır (buzdağı küpü).

## 6. Simpson paradoksu: Berkeley verisi

**Konu.** **Simpson paradoksu**: toplamda görülen bir eğilimin, verinin alt gruplarında ortadan kalkması ya da tersine dönmesi. Roll-up'ın en tehlikeli sonucudur.

**Berkeley, 1973.** Kaliforniya Üniversitesi Berkeley'in en büyük altı bölümüne (A–F) yapılan yüksek lisans başvuruları; erkek (E) ve kadın (K) başvuru ve kabul sayıları (Bickel, Hammel ve O'Connell, 1975, *Science*, 187). Toplamda erkeklerin kabul oranı kadınlarınkinden belirgin biçimde yüksek. Bölüm bölüm bakınca ne oluyor?

🐍 **Python:** Sözlükten tablo, 12 satır.
* `list("AABBCCDDEEFF")`: metni harflerine ayırıp listeye çevirir: `["A", "A", "B", …]`.
* `["E", "K"] * 6`: listeyi 6 kez arka arkaya ekler (listeyle çarpma **tekrar** demektir; numpy dizisinde ise sayıları çarpardı).

📊 **Konu:** Her satır bir (bölüm, cinsiyet) çifti: kaç başvuru, kaç kabul.

In [ ]:
b = pd.DataFrame({
    "bolum":    list("AABBCCDDEEFF"),
    "cinsiyet": ["E", "K"] * 6,
    "basvuru":  [825, 108, 560, 25, 325, 593, 417, 375, 191, 393, 373, 341],
    "kabul":    [512, 89, 353, 17, 120, 202, 138, 131, 53, 94, 22, 24],
})

🔍 **Ara adım:** Tabloyu görelim: her bölüm için önce erkek, sonra kadın satırı.

In [ ]:
b

🐍 **Python:** Kalıp, bu kez **iki ölçü sütunuyla**: cinsiyete göre grupla → `[["basvuru", "kabul"]]` (çift köşeli parantez: iki sütun birlikte) → her grubun **toplamı**. Sonuç iki satırlı (E, K), iki sütunlu bir tablo.

📊 **Konu:** **Roll-up:** bölüm boyutu kaldırılıyor; üniversite toplamı.

In [ ]:
toplam = b.groupby("cinsiyet")[["basvuru", "kabul"]].sum()

🔍 **Ara adım:** Erkek: 2691 başvuru, 1198 kabul. Kadın: 1835 başvuru, 557 kabul.

In [ ]:
print(toplam)

🐍 **Python:** İki sütun birbirine bölünür: satır satır, **aynı etiketler** (E ile E, K ile K) eşleşerek.

📊 **Konu:** Dikkat: oranı, oranların ortalaması olarak değil, **toplam kabul ÷ toplam başvuru** olarak hesaplıyoruz (oran cebirsel bir ölçüdür; 2. bölüm).

In [ ]:
print((toplam["kabul"] / toplam["basvuru"]).round(3))              # roll-up: üniversite

🐍 **Python:** Satır satır bölme; sonuç `b`'ye yeni bir sütun (`oran`) olarak eklenir.

📊 **Konu:** Her (bölüm, cinsiyet) çiftinin kabul oranı.

In [ ]:
b["oran"] = (b["kabul"] / b["basvuru"]).round(3)

🐍 **Python:** `pivot`, `pivot_table`'ın **özetlemeyen** kardeşidir: satırlara bölüm, sütunlara cinsiyet koyar ve hücreye `oran` değerini olduğu gibi yazar. Her (bölüm, cinsiyet) çiftinde zaten tek bir değer olduğu için özet işlemi gerekmez.

📊 **Konu:** **Drill-down:** aynı karşılaştırma bölüm bölüm.

In [ ]:
print(b.pivot(index="bolum", columns="cinsiyet", values="oran"))   # drill-down: bölüm

✅ **Sonucu okuyalım:** Toplamda erkekler **0,445**, kadınlar **0,304**. Bölüm bölüm ise A, B, D ve F'de kadınların oranı **daha yüksek** (A'da 0,824 ile 0,621); C ve E'de biraz daha düşük.

Toplamdaki fark nereden geliyor? Kadınların başvurularının büyük kısmı kabul oranı düşük C ve E bölümlerine gitmiş (593 + 393 başvuru); erkekler ise çoğunlukla kabul oranı yüksek A ve B'ye başvurmuş (825 + 560). Toplam oran, **bölüm seçimini** cinsiyet etkisi gibi gösteriyor. Karıştırılan boyut (bölüm) hem gruplarla (cinsiyet) hem sonuçla (kabul) ilişkili olduğu için roll-up yanlış bir hikâye anlatıyor.

**Proje 1, 3. bölüm:** kendi verinizde toplamda gördüğünüz bir eğilimin alt gruplarda değişip değişmediğine bakın.

🔍 **Ara adım:** **Bir tuzak:** oranların düz ortalaması (her bölüme eşit ağırlık) erkekte 0,381, kadında 0,417 çıkar: bu da başka bir sayı! Hangi özetin hangi soruya cevap verdiğini bilmek gerekir. "Bir başvurunun kabul edilme olasılığı" için toplam ÷ toplam; "tipik bir bölümde kabul oranı" için bölüm ortalaması.

In [ ]:
b.groupby("cinsiyet")["oran"].mean().round(3)

## 7. Ek (ders notundan): zaman noktasına göre birleştirme

**Konu.** Ambardan model eğitimi için veri üretirken her örneğin öznitelikleri, **tahmin anında bilinen hâliyle** alınmalıdır (**zaman noktası doğruluğu**). Ambarda değişen boyut değerleri silinmeden, geçerlilik tarihleriyle saklanır (**yavaş değişen boyut, tip 2**). Örnek: 17 numaralı müşterinin gelir grubu 2021'den beri "orta", 1 Temmuz 2024'ten itibaren "düşük". İki başvurusu var: Mart 2024 ve Şubat 2025. Her başvuru, **kendi tarihinde geçerli** gelir grubunu almalı.

🐍 **Python:** `pd.to_datetime` metni **tarih** türüne çevirir; tarihler ancak böyle karşılaştırılabilir ve sıralanabilir.

📊 **Konu:** Olgu tarafı: iki kredi başvurusu.

In [ ]:
basvuru = pd.DataFrame({"musteri_id": [17, 17],
                        "tarih": pd.to_datetime(["2024-03-15", "2025-02-01"])})

🐍 **Python:** Aynı kalıp.

📊 **Konu:** Boyut tarafı (tip 2): her satır, gelir grubunun **geçerli olmaya başladığı** tarihi taşıyor.

In [ ]:
musteri = pd.DataFrame({"musteri_id": [17, 17],
                        "baslangic": pd.to_datetime(["2021-01-01", "2024-07-01"]),
                        "gelir_grubu": ["orta", "düşük"]})

🔍 **Ara adım:** **Yanlış yol:** sıradan `merge` yalnız müşteri numarasına bakar; her başvuru **iki satırla da** eşleşir (4 satır). Yalnız güncel satır tutulsaydı, Mart 2024 başvurusu yanlışlıkla "düşük" alırdı: başvurudan **sonra** olan bir değişiklik modele sızardı.

In [ ]:
basvuru.merge(musteri, on="musteri_id")

🐍 **Python:** `pd.merge_asof`: her başvuru için, başvuru tarihinden **önceki ya da aynı günkü en son** boyut satırını bulur.
* `left_on="tarih"`, `right_on="baslangic"`: hangi tarih sütunlarının karşılaştırılacağı.
* `by="musteri_id"`: aramayı aynı müşterinin satırlarıyla sınırlar.
* `sort_values(...)`: tabloyu bir sütuna göre sıralar; `merge_asof` iki tablonun da tarihe göre **sıralı** olmasını ister.

📊 **Konu:** **Doğru yol:** zaman noktasına göre birleştirme.

In [ ]:
egitim = pd.merge_asof(basvuru.sort_values("tarih"), musteri.sort_values("baslangic"),
                       left_on="tarih", right_on="baslangic", by="musteri_id")

🐍 **Python:** Yalnız üç sütunu gösterir.

In [ ]:
print(egitim[["musteri_id", "tarih", "gelir_grubu"]])

✅ **Sonucu okuyalım:** Mart 2024 başvurusu **orta**, Şubat 2025 başvurusu **düşük** aldı: her biri kendi tarihinde geçerli olan değeri. SQL'de aynı iş tarih koşullu bir JOIN ile yapılır. Bölmeyi de zamana göre yapmak gerekir: eski başvurularla eğitip yenilerle test etmek.